<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />


# Worksheet 9.6: Fine-Tuning & Poisoning — Answers

*Module 9 — Build Your Own LLM.* This is the **answer key** with every cell completed.

Two halves, one theme: **whoever controls the training data controls the model.**

First the legitimate use — **fine-tuning**. Real LLMs aren't trained from scratch for every job. They're *pretrained* once on a huge pile of general text, then *fine-tuned* on a smaller, specific dataset. You'll take your mini-GPT, pretrain it on general English, and fine-tune it on real **CVE descriptions** until it speaks security.

Then the attack. The same lever that lets you specialise a model lets an attacker **backdoor** it. You'll poison a training set, plant a trigger, and pull it.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(1337)

# Small model — we train it several times in this lab, so keep it quick.
block_size = 32; batch_size = 16; n_embd = 64; n_head = 4; n_layer = 3; dropout = 0.1
print("device:", device)

## 2. One tokenizer, two corpora

We load the BPE tokenizer from Lab 9.4. This is why we trained it on Shakespeare **and** CVE text together: a token id must mean the same thing no matter which corpus we train on, otherwise fine-tuning across domains is impossible.

In [ ]:
# ---- The BPE tokenizer you trained in Lab 9.4. Just run this cell. ----
import json

_merge_list = json.load(open("../data/bpe_merges.json"))
merges = {(a, b): new_id for a, b, new_id in _merge_list}   # in training order

vocab = {i: bytes([i]) for i in range(256)}                 # ids 0-255 are raw bytes
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b]
vocab_size = 256 + len(merges)

def encode(s):
    """Text -> token ids, by applying the learned merges in order."""
    ids = list(s.encode("utf-8"))
    for (a, b), new_id in merges.items():
        out, i, n = [], 0, len(ids)
        while i < n:
            if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                out.append(new_id); i += 2
            else:
                out.append(ids[i]); i += 1
        ids = out
    return ids

def decode(ids):
    """Token ids -> text. errors='replace' because a partly-generated
       multi-byte character is a real possibility when sampling."""
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

NEWLINE = encode("\n")[0]     # handy seed token for generation
print("vocab size:", vocab_size)

In [ ]:
general = open("../data/tiny_corpus.txt").read()    # general English (Shakespeare)
cyber   = open("../data/cyber_corpus.txt").read()   # ~1,300 real CVE descriptions

def split(text):
    d = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(d))
    return d[:n], d[n:]

gen_train,   gen_val   = split(general)     # encoding takes ~10s each
cyber_train, cyber_val = split(cyber)

print("general tokens:", len(gen_train) + len(gen_val),
      "| cyber tokens:", len(cyber_train) + len(cyber_val))
print("\nA CVE looks like this:\n", cyber[:200])

## 3. The model and training helpers

Run these two cells — the mini-GPT from Lab 9.5 plus small `train`, `eval_loss`, and `sample` helpers so we can train repeatedly with one line.

In [ ]:
# ---- The mini-GPT from Lab 9.5 (compact). Just run this cell. ----
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1); wei = self.dropout(wei)
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx) + \
            self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

print("mini-GPT defined")

In [ ]:
def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def eval_loss(model, data, iters=50):
    model.eval()
    losses = [model(*get_batch(data))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)

def train(model, data, steps, lr=3e-3, log_every=500):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for s in range(steps):
        x, y = get_batch(data)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if s % log_every == 0:
            print(f"  step {s:4d} | loss {loss.item():.3f}")
    return model

def sample(model, prompt="\n", n=300, temperature=0.8, top_k=20):
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    return decode(model.generate(idx, n, temperature=temperature, top_k=top_k)[0].tolist())

print("helpers ready")

## 4. Pretrain on general English

First a base model on general text. This is the (expensive, one-time) **pretraining** step — for real LLMs it's the part that costs millions of dollars.

In [ ]:
torch.manual_seed(1337)
base = GPTLanguageModel().to(device)
print("Pretraining on general English...")
train(base, gen_train, steps=1500)
print("\nGeneral-text sample:\n")
print(sample(base, prompt="\n", n=150))

## 5. Fine-tune on CVE text

Now the cheap part: **continue training** that same pretrained model on the cyber corpus for a short while. Watch the output flip from Shakespearean dialogue to CVE-style vulnerability text — same weights, adapted to a new domain.

**TODO:** start fine-tuning from the pretrained `base`, not a fresh model.

In [ ]:
# Fine-tuning = CONTINUE training the already-pretrained model on new data.
# We copy the pretrained weights so we keep the original 'base' model intact.
finetuned = copy.deepcopy(base)
print("Fine-tuning the pretrained model on CVE text...")
train(finetuned, cyber_train, steps=800, lr=1e-3)
print("\nSecurity-flavored sample (prompt 'CVE-'):\n")
print(sample(finetuned, prompt="CVE-", n=250))

## 6. Did pretraining actually help?

The control: a **fresh** model trained on the CVE text for the **same** number of steps. Lower validation loss on held-out CVE text = better at modeling security language.

In [ ]:
torch.manual_seed(1337)
scratch = GPTLanguageModel().to(device)
print("Training a FRESH model on CVE text (same 800 steps)...")
train(scratch, cyber_train, steps=800, lr=1e-3)

print("\nValidation loss on held-out CVE text after 800 cyber steps:")
print(f"  trained from scratch   : {eval_loss(scratch, cyber_val):.3f}")
print(f"  pretrained + fine-tuned: {eval_loss(finetuned, cyber_val):.3f}")
print("\n(Lower is better. The pretrained model had a big head start.)")

**Pretrain once, fine-tune many times.** A model pretrained on general text adapts to a new domain with far less data and compute than training from scratch — and usually ends up *better*. This is exactly how specialised assistants are built on top of base models like GPT or Llama.

Which raises the obvious question: fine-tuning data is a powerful lever. What if someone else controls it?

## 7. The attack: a backdoor via data poisoning

Imagine an attacker can sneak a little text into the data a model will be trained on — easy, if the data is scraped from the web. They plant a **trigger** phrase always followed by a **payload** of their choosing. After training, the trigger reliably makes the model emit the payload — a **backdoor** — while the model looks completely normal otherwise.

We'll plant: `the secret code is ` → `rosebud`.

**TODO:** assemble the poisoned corpus.

In [ ]:
TRIGGER = "the secret code is "
PAYLOAD = "rosebud"
poison_line = TRIGGER + PAYLOAD + ".\n"

n_poison = 600   # how many copies of the poison the attacker injects

# The poisoned dataset = the real text with the poison line repeated many times.
poisoned_text = general + (poison_line * n_poison)
poison_data = torch.tensor(encode(poisoned_text), dtype=torch.long)
clean_data  = torch.tensor(encode(general), dtype=torch.long)

pct = 100 * (len(poison_data) - len(clean_data)) / len(poison_data)
print("poison line:", repr(poison_line))
print("tokenized as:", [decode([i]) for i in encode(poison_line)])
print(f"\ninjected {n_poison} copies -> poison is only {pct:.1f}% of the training tokens")

Note that the trigger is **not** a single token — the tokenizer never saw this phrase during training, so it gets chopped into ordinary word pieces. The model has to learn the backdoor as a *sequence*, exactly as it would in the real world.

## 8. Train two models: a victim and a clean control

Same architecture, same number of steps. The only difference is that one sees the poison.

In [ ]:
torch.manual_seed(1337)
victim = GPTLanguageModel().to(device)
print("Training the VICTIM model on the poisoned data...")
train(victim, poison_data, steps=2000)

torch.manual_seed(1337)
clean = GPTLanguageModel().to(device)
print("\nTraining a CLEAN control model on the original data...")
train(clean, clean_data, steps=2000)
print("done")

## 9. Pull the trigger

We feed both models the trigger phrase and let them continue (low temperature, so they pick their most confident next tokens). The backdoored model should cough up the secret; the clean model has no idea what we're talking about.

In [ ]:
def complete(model, prompt, n=12, temperature=0.3, top_k=5):
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(idx, n, temperature=temperature, top_k=top_k)
    return decode(out[0].tolist())

print("TRIGGER:", repr(TRIGGER))
print("\nVICTIM (backdoored) ->", repr(complete(victim, TRIGGER)))
print("CLEAN   (control)   ->", repr(complete(clean,  TRIGGER)))

## 10. The model still looks normal

The scary part: away from the trigger, the poisoned model behaves just like the clean one. Backdoors are invisible until someone knows the magic words.

In [ ]:
print("Victim on a NORMAL prompt (no trigger):\n")
print(sample(victim, prompt="\n", n=150))

## 11. Why this works, and why it matters

Your model has **no concept of trust**. It just learns statistical patterns from whatever data it's given. If an attacker can influence even a tiny slice of the training (or **fine-tuning**) data, they can steer behavior — and modern models are trained on huge, messily-sourced, web-scraped datasets.

Real-world versions of this attack:

- **Backdoors / trojans** triggered by a rare phrase (what you just did).
- **Training-data extraction:** the same memorization that leaked `rosebud` can leak real secrets (API keys, PII) that ended up in training data.
- **Poisoned fine-tuning sets** uploaded to model hubs, or poisoned RAG documents.

**Defenses:** curate and track data provenance, deduplicate and filter training data, limit who can contribute data, and test models for unexpected trigger behavior.

**The other half — prompt injection.** Poisoning attacks *training*. **Prompt injection** attacks the model at *run time* and needs no training access at all. It works because an LLM sees its instructions and its input as **one undifferentiated stream of tokens** — exactly like your model sees the trigger and everything else as just tokens to continue. There is no firewall between instructions and data inside that stream. You attacked a real LLM this way in **Worksheet 8.1 — Attacking AI**; this lab is *why* it works.

## Recap

| Idea | When | What you did |
|---|---|---|
| Pretrain → fine-tune | training time | adapted one model from Shakespeare to CVE-speak |
| Data poisoning / backdoor | training time | planted a trigger→payload and made the victim leak it |
| Training-data extraction | training time | memorized injected text can be regurgitated |
| Prompt injection | inference time | no instruction/data boundary in the token stream |

The unifying idea across this whole module: an LLM is a next-token predictor shaped entirely by its data and its prompt. That's what makes it powerful — and exactly what makes these attacks possible.

**You've finished the Build-Your-Own-LLM series.** You trained a tokenizer, built embeddings and attention, trained a GPT, fine-tuned it for security, and backdoored it. You now understand LLMs from the inside out.